# 07 — Model Dataset Preparation

This notebook prepares the final leakage-safe dataset for supervised severe-rainfall forecasting.

The dataset combines:

- current meteorological observations;
- temporal cyclical features;
- lagged meteorological observations;
- recent precipitation accumulation;
- short-term atmospheric changes;
- severe-rainfall targets for forecast horizons from +1h to +6h.

The temporal split is intentionally chronological:

- Training period: 2018–2020
- Test period: 2021

The test period is never used during model training.

Rows affected by missing meteorological observations, missing historical lags, or unavailable future targets are excluded only for the corresponding modeling task.

No missing meteorological values are imputed during this stage.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# 1. CONFIGURATION
# ============================================================

INPUT_PATH = Path(
    "../data/processed/recife_2018_2021_targets.csv"
)

OUTPUT_PATH = Path(
    "../data/processed/recife_2018_2021_model_dataset.csv"
)

TEST_START = pd.Timestamp(
    "2021-01-01 00:00",
    tz="UTC"
)

FORECAST_HORIZONS = [
    1,
    2,
    3,
    4,
    5,
    6,
]


# ============================================================
# 2. LOAD TARGET DATASET
# ============================================================

if not INPUT_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {INPUT_PATH}"
    )


df = pd.read_csv(
    INPUT_PATH,
    parse_dates=["datetime_utc"],
)


df = (
    df
    .set_index("datetime_utc")
    .sort_index()
)


print("=== DATASET LOADED ===")
print()

print(
    "Start:",
    df.index.min()
)

print(
    "End:",
    df.index.max()
)

print(
    "Rows:",
    len(df)
)


# ============================================================
# 3. VALIDATE TARGET COLUMNS
# ============================================================

target_columns = [
    f"target_{horizon}h"
    for horizon in FORECAST_HORIZONS
]


missing_target_columns = [
    column
    for column in target_columns
    if column not in df.columns
]


if missing_target_columns:
    raise ValueError(
        "Missing target columns: "
        f"{missing_target_columns}"
    )


for column in target_columns:

    df[column] = (
        pd.to_numeric(
            df[column],
            errors="coerce",
        )
        .astype("Int64")
    )


# ============================================================
# 4. TEMPORAL FEATURES
# ============================================================

df["hour"] = df.index.hour

df["day_of_year"] = (
    df.index.dayofyear
)


df["hour_sin"] = np.sin(
    2
    * np.pi
    * df["hour"]
    / 24
)

df["hour_cos"] = np.cos(
    2
    * np.pi
    * df["hour"]
    / 24
)


df["day_of_year_sin"] = np.sin(
    2
    * np.pi
    * df["day_of_year"]
    / 365.25
)

df["day_of_year_cos"] = np.cos(
    2
    * np.pi
    * df["day_of_year"]
    / 365.25
)


# ============================================================
# 5. WIND DIRECTION AS A CYCLICAL VARIABLE
# ============================================================

wind_direction_rad = np.deg2rad(
    df["wind_direction_deg"]
)


df["wind_direction_sin"] = np.sin(
    wind_direction_rad
)

df["wind_direction_cos"] = np.cos(
    wind_direction_rad
)


# ============================================================
# 6. PRECIPITATION LAGS
# ============================================================

for lag in [
    1,
    2,
    3,
    6,
]:

    df[
        f"precipitation_lag_{lag}h"
    ] = (
        df["precipitation_mm"]
        .shift(lag)
    )


# ============================================================
# 7. METEOROLOGICAL LAGS
# ============================================================

lag_variables = [
    "temperature_c",
    "humidity_pct",
    "pressure_station_mb",
    "wind_speed_ms",
]


for variable in lag_variables:

    df[
        f"{variable}_lag_1h"
    ] = (
        df[variable]
        .shift(1)
    )


# ============================================================
# 8. RECENT PRECIPITATION ACCUMULATION
# ============================================================

df["precipitation_sum_3h"] = (
    df["precipitation_mm"]
    .rolling(
        window=3,
        min_periods=3,
    )
    .sum()
)


df["precipitation_sum_6h"] = (
    df["precipitation_mm"]
    .rolling(
        window=6,
        min_periods=6,
    )
    .sum()
)


# ============================================================
# 9. SHORT-TERM ATMOSPHERIC CHANGES
# ============================================================

df["temperature_change_1h"] = (
    df["temperature_c"]
    - df["temperature_c"].shift(1)
)


df["humidity_change_1h"] = (
    df["humidity_pct"]
    - df["humidity_pct"].shift(1)
)


df["pressure_change_1h"] = (
    df["pressure_station_mb"]
    - df["pressure_station_mb"].shift(1)
)


# ============================================================
# 10. FINAL MODEL FEATURE SET
# ============================================================

feature_columns = [

    # Current meteorological state
    "precipitation_mm",
    "temperature_c",
    "humidity_pct",
    "pressure_station_mb",
    "wind_speed_ms",
    "wind_gust_ms",

    # Cyclical temporal representation
    "hour_sin",
    "hour_cos",
    "day_of_year_sin",
    "day_of_year_cos",

    # Wind direction
    "wind_direction_sin",
    "wind_direction_cos",

    # Precipitation history
    "precipitation_lag_1h",
    "precipitation_lag_2h",
    "precipitation_lag_3h",
    "precipitation_lag_6h",
    "precipitation_sum_3h",
    "precipitation_sum_6h",

    # Meteorological history
    "temperature_c_lag_1h",
    "humidity_pct_lag_1h",
    "pressure_station_mb_lag_1h",
    "wind_speed_ms_lag_1h",

    # Recent changes
    "temperature_change_1h",
    "humidity_change_1h",
    "pressure_change_1h",
]


# ============================================================
# 11. DATA LEAKAGE PROTECTION
# ============================================================

for feature in feature_columns:

    feature_lower = (
        feature.lower()
    )

    if (
        "future_" in feature_lower
        or "target_" in feature_lower
    ):
        raise ValueError(
            f"Future information detected "
            f"in feature set: {feature}"
        )


print()
print(
    "Leakage check passed."
)

print(
    "Number of model features:",
    len(feature_columns)
)


# ============================================================
# 12. FEATURE COMPLETENESS
# ============================================================

df["features_complete"] = (
    df[feature_columns]
    .notna()
    .all(axis=1)
)


complete_features = int(
    df["features_complete"].sum()
)

incomplete_features = (
    len(df)
    - complete_features
)


print()
print(
    "=== FEATURE AVAILABILITY ==="
)

print(
    "Rows with complete features:",
    complete_features
)

print(
    "Rows with incomplete features:",
    incomplete_features
)

print(
    "Feature completeness:",
    f"{complete_features / len(df) * 100:.2f}%"
)


# ============================================================
# 13. TEMPORAL TRAIN / TEST SPLIT
# ============================================================

df["split"] = np.where(
    df.index < TEST_START,
    "train",
    "test",
)


print()
print(
    "=== TEMPORAL SPLIT ==="
)

print(
    "Training period:",
    df[
        df["split"] == "train"
    ].index.min(),
    "to",
    df[
        df["split"] == "train"
    ].index.max(),
)

print(
    "Test period:",
    df[
        df["split"] == "test"
    ].index.min(),
    "to",
    df[
        df["split"] == "test"
    ].index.max(),
)


# ============================================================
# 14. CREATE HORIZON-SPECIFIC ELIGIBILITY FLAGS
# ============================================================

for horizon in FORECAST_HORIZONS:

    target_column = (
        f"target_{horizon}h"
    )

    eligibility_column = (
        f"eligible_{horizon}h"
    )


    df[eligibility_column] = (
        df["features_complete"]
        &
        df[target_column].notna()
    )


# ============================================================
# 15. MODEL DATASET SUMMARY
# ============================================================

summary_rows = []


for horizon in FORECAST_HORIZONS:

    target_column = (
        f"target_{horizon}h"
    )

    eligibility_column = (
        f"eligible_{horizon}h"
    )


    for split_name in [
        "train",
        "test",
    ]:

        subset = df[
            (df["split"] == split_name)
            &
            df[eligibility_column]
        ]


        positives = int(
            (
                subset[target_column]
                == 1
            )
            .sum()
        )


        negatives = int(
            (
                subset[target_column]
                == 0
            )
            .sum()
        )


        total = len(
            subset
        )


        summary_rows.append(
            {
                "horizon_hours":
                    horizon,

                "split":
                    split_name,

                "samples":
                    total,

                "positive_samples":
                    positives,

                "negative_samples":
                    negatives,

                "positive_pct":
                    (
                        positives
                        / total
                        * 100
                        if total > 0
                        else np.nan
                    ),
            }
        )


model_summary_df = pd.DataFrame(
    summary_rows
)


print()
print(
    "=== MODEL DATASET SUMMARY ==="
)

display(
    model_summary_df.round(3)
)


# ============================================================
# 16. POSITIVE SAMPLES BY HORIZON AND SPLIT
# ============================================================

positive_pivot = (
    model_summary_df
    .pivot(
        index="horizon_hours",
        columns="split",
        values="positive_samples",
    )
)


print()
print(
    "=== POSITIVE SAMPLES BY SPLIT ==="
)

display(
    positive_pivot
)


# ============================================================
# 17. SAVE FINAL MODEL DATASET
# ============================================================

columns_to_save = [

    *feature_columns,

    *target_columns,

    "features_complete",
    "split",

    *[
        f"eligible_{horizon}h"
        for horizon
        in FORECAST_HORIZONS
    ],
]


df[
    columns_to_save
].to_csv(
    OUTPUT_PATH,
    index=True,
)


print()
print(
    "Model dataset saved to:"
)

print(
    OUTPUT_PATH
)

=== DATASET LOADED ===

Start: 2018-01-01 00:00:00+00:00
End: 2021-11-12 06:00:00+00:00
Rows: 33871

Leakage check passed.
Number of model features: 25

=== FEATURE AVAILABILITY ===
Rows with complete features: 21321
Rows with incomplete features: 12550
Feature completeness: 62.95%

=== TEMPORAL SPLIT ===
Training period: 2018-01-01 00:00:00+00:00 to 2020-12-31 23:00:00+00:00
Test period: 2021-01-01 00:00:00+00:00 to 2021-11-12 06:00:00+00:00

=== MODEL DATASET SUMMARY ===


,horizon_hours,split,samples,positive_samples,negative_samples,positive_pct
0,1,train,14036,42,13994,0.299
1,1,test,7246,38,7208,0.524
2,2,train,14021,42,13979,0.300
3,2,test,7244,38,7206,0.525
4,3,train,14006,42,13964,0.300
5,3,test,7242,38,7204,0.525
6,4,train,13993,42,13951,0.300
7,4,test,7240,38,7202,0.525
8,5,train,13981,41,13940,0.293
9,5,test,7237,38,7199,0.525



=== POSITIVE SAMPLES BY SPLIT ===


split,test,train
horizon_hours,,
1,38,42
2,38,42
3,38,42
4,38,42
5,38,41
6,38,40



Model dataset saved to:
..\data\processed\recife_2018_2021_model_dataset.csv


## Conclusion

The final supervised-learning dataset was prepared using only meteorological information available at or before each prediction timestamp.

Temporal, lagged, rolling and short-term atmospheric features were generated without using future precipitation information.

A chronological train/test strategy was adopted:

- 2018–2020: training data
- 2021: test data

Rows with incomplete model features or unavailable horizon-specific targets are excluded from the corresponding modeling task.

No random temporal shuffling, missing-value imputation, feature scaling or resampling was performed during this stage. These operations, when required, will be fitted exclusively on the training data inside the modeling pipeline.

The dataset is now ready for baseline severe-rainfall forecasting models.

In [2]:
# ============================================================
# 18. FEATURE AVAILABILITY AUDIT
# ============================================================

feature_audit = []


for feature in feature_columns:

    train_data = df[
        df["split"] == "train"
    ]

    test_data = df[
        df["split"] == "test"
    ]


    train_available = int(
        train_data[feature]
        .notna()
        .sum()
    )

    test_available = int(
        test_data[feature]
        .notna()
        .sum()
    )


    train_missing = (
        len(train_data)
        - train_available
    )

    test_missing = (
        len(test_data)
        - test_available
    )


    feature_audit.append(
        {
            "feature":
                feature,

            "train_missing":
                train_missing,

            "train_missing_pct":
                (
                    train_missing
                    / len(train_data)
                    * 100
                ),

            "test_missing":
                test_missing,

            "test_missing_pct":
                (
                    test_missing
                    / len(test_data)
                    * 100
                ),
        }
    )


feature_audit_df = (
    pd.DataFrame(
        feature_audit
    )
    .sort_values(
        "train_missing_pct",
        ascending=False,
    )
    .reset_index(drop=True)
)


print(
    "=== FEATURE MISSINGNESS BY SPLIT ==="
)

display(
    feature_audit_df.round(2)
)


# ============================================================
# 19. FEATURE MISSINGNESS ON SEVERE TRAINING EXAMPLES
# ============================================================

positive_feature_audit = []


for horizon in FORECAST_HORIZONS:

    target_column = (
        f"target_{horizon}h"
    )


    positive_train_rows = df[
        (df["split"] == "train")
        &
        (df[target_column] == 1)
    ]


    original_positive_count = len(
        positive_train_rows
    )


    for feature in feature_columns:

        missing_on_positive = int(
            positive_train_rows[
                feature
            ]
            .isna()
            .sum()
        )


        positive_feature_audit.append(
            {
                "horizon_hours":
                    horizon,

                "feature":
                    feature,

                "total_positive_rows":
                    original_positive_count,

                "positive_rows_missing_feature":
                    missing_on_positive,
            }
        )


positive_feature_audit_df = pd.DataFrame(
    positive_feature_audit
)


positive_loss_pivot = (
    positive_feature_audit_df
    .pivot(
        index="feature",
        columns="horizon_hours",
        values="positive_rows_missing_feature",
    )
)


positive_loss_pivot[
    "max_positive_rows_lost"
] = (
    positive_loss_pivot.max(
        axis=1
    )
)


positive_loss_pivot = (
    positive_loss_pivot
    .sort_values(
        "max_positive_rows_lost",
        ascending=False,
    )
)


print()
print(
    "=== POSITIVE TRAINING SAMPLES LOST BY FEATURE ==="
)

display(
    positive_loss_pivot
)

=== FEATURE MISSINGNESS BY SPLIT ===


,feature,train_missing,train_missing_pct,test_missing,test_missing_pct
0,wind_gust_ms,12071,45.89,16,0.21
1,wind_speed_ms_lag_1h,12036,45.76,17,0.22
2,wind_speed_ms,12035,45.75,17,0.22
3,wind_direction_sin,12035,45.75,17,0.22
4,wind_direction_cos,12035,45.75,17,0.22
5,precipitation_sum_6h,655,2.49,158,2.09
6,precipitation_sum_3h,559,2.13,109,1.44
7,temperature_change_1h,527,2.00,32,0.42
8,humidity_change_1h,527,2.00,167,2.21
9,pressure_change_1h,527,2.00,32,0.42



=== POSITIVE TRAINING SAMPLES LOST BY FEATURE ===


horizon_hours,1,2,3,4,5,6,max_positive_rows_lost
feature,,,,,,,
wind_gust_ms,28,28,28,28,29,30,30
wind_speed_ms_lag_1h,28,28,28,29,29,30,30
wind_direction_sin,27,28,28,28,29,29,29
wind_direction_cos,27,28,28,28,29,29,29
wind_speed_ms,27,28,28,28,29,29,29
precipitation_lag_6h,1,1,1,1,2,2,2
precipitation_sum_6h,1,1,1,1,1,2,2
precipitation_lag_2h,0,0,1,1,1,1,1
humidity_change_1h,0,0,0,1,1,1,1


In [3]:
# ============================================================
# 20. FINAL FEATURE SET
# ============================================================

FINAL_FEATURE_COLUMNS = [

    # Current meteorological state
    "precipitation_mm",
    "temperature_c",
    "humidity_pct",
    "pressure_station_mb",

    # Cyclical temporal representation
    "hour_sin",
    "hour_cos",
    "day_of_year_sin",
    "day_of_year_cos",

    # Precipitation history
    "precipitation_lag_1h",
    "precipitation_lag_2h",
    "precipitation_lag_3h",
    "precipitation_lag_6h",
    "precipitation_sum_3h",
    "precipitation_sum_6h",

    # Meteorological history
    "temperature_c_lag_1h",
    "humidity_pct_lag_1h",
    "pressure_station_mb_lag_1h",

    # Recent atmospheric changes
    "temperature_change_1h",
    "humidity_change_1h",
    "pressure_change_1h",
]


# ============================================================
# 21. FINAL LEAKAGE CHECK
# ============================================================

for feature in FINAL_FEATURE_COLUMNS:

    feature_lower = feature.lower()

    if (
        "future_" in feature_lower
        or "target_" in feature_lower
    ):
        raise ValueError(
            f"Future information detected: {feature}"
        )


print("Final leakage check passed.")
print(
    "Final number of model features:",
    len(FINAL_FEATURE_COLUMNS)
)


# ============================================================
# 22. RECOMPUTE FEATURE COMPLETENESS
# ============================================================

df["features_complete_final"] = (
    df[FINAL_FEATURE_COLUMNS]
    .notna()
    .all(axis=1)
)


complete_final = int(
    df["features_complete_final"]
    .sum()
)

incomplete_final = (
    len(df)
    - complete_final
)


print()
print(
    "=== FINAL FEATURE AVAILABILITY ==="
)

print(
    "Rows with complete features:",
    complete_final
)

print(
    "Rows with incomplete features:",
    incomplete_final
)

print(
    "Feature completeness:",
    f"{complete_final / len(df) * 100:.2f}%"
)


# ============================================================
# 23. RECOMPUTE ELIGIBILITY FOR EACH HORIZON
# ============================================================

for horizon in FORECAST_HORIZONS:

    target_column = (
        f"target_{horizon}h"
    )

    eligibility_column = (
        f"eligible_final_{horizon}h"
    )


    df[eligibility_column] = (
        df["features_complete_final"]
        &
        df[target_column].notna()
    )


# ============================================================
# 24. FINAL MODEL DATASET SUMMARY
# ============================================================

final_summary_rows = []


for horizon in FORECAST_HORIZONS:

    target_column = (
        f"target_{horizon}h"
    )

    eligibility_column = (
        f"eligible_final_{horizon}h"
    )


    for split_name in [
        "train",
        "test",
    ]:

        subset = df[
            (df["split"] == split_name)
            &
            df[eligibility_column]
        ]


        positives = int(
            (
                subset[target_column]
                == 1
            )
            .sum()
        )

        negatives = int(
            (
                subset[target_column]
                == 0
            )
            .sum()
        )

        total = len(
            subset
        )


        final_summary_rows.append(
            {
                "horizon_hours":
                    horizon,

                "split":
                    split_name,

                "samples":
                    total,

                "positive_samples":
                    positives,

                "negative_samples":
                    negatives,

                "positive_pct":
                    (
                        positives
                        / total
                        * 100
                        if total > 0
                        else np.nan
                    ),
            }
        )


final_model_summary_df = pd.DataFrame(
    final_summary_rows
)


print()
print(
    "=== FINAL MODEL DATASET SUMMARY ==="
)

display(
    final_model_summary_df.round(3)
)


# ============================================================
# 25. POSITIVE SAMPLES AFTER FEATURE REDUCTION
# ============================================================

final_positive_pivot = (
    final_model_summary_df
    .pivot(
        index="horizon_hours",
        columns="split",
        values="positive_samples",
    )
)


print()
print(
    "=== FINAL POSITIVE SAMPLES BY SPLIT ==="
)

display(
    final_positive_pivot
)


# ============================================================
# 26. COMPARE OLD VS FINAL FEATURE SET
# ============================================================

comparison = pd.DataFrame(
    {
        "metric": [
            "number_of_features",
            "complete_rows",
            "incomplete_rows",
            "complete_pct",
        ],

        "original_25_features": [
            len(feature_columns),
            int(df["features_complete"].sum()),
            int((~df["features_complete"]).sum()),
            df["features_complete"].mean() * 100,
        ],

        "final_20_features": [
            len(FINAL_FEATURE_COLUMNS),
            int(df["features_complete_final"].sum()),
            int((~df["features_complete_final"]).sum()),
            df["features_complete_final"].mean() * 100,
        ],
    }
)


print()
print(
    "=== FEATURE SET COMPARISON ==="
)

display(
    comparison.round(2)
)


# ============================================================
# 27. SAVE FINAL MODEL DATASET
# ============================================================

FINAL_OUTPUT_PATH = Path(
    "../data/processed/recife_2018_2021_model_dataset.csv"
)


columns_to_save = [

    *FINAL_FEATURE_COLUMNS,

    *target_columns,

    "features_complete_final",
    "split",

    *[
        f"eligible_final_{horizon}h"
        for horizon
        in FORECAST_HORIZONS
    ],
]


df[
    columns_to_save
].to_csv(
    FINAL_OUTPUT_PATH,
    index=True,
)


print()
print(
    "Final model dataset saved to:"
)

print(
    FINAL_OUTPUT_PATH
)

Final leakage check passed.
Final number of model features: 20

=== FINAL FEATURE AVAILABILITY ===
Rows with complete features: 32881
Rows with incomplete features: 990
Feature completeness: 97.08%

=== FINAL MODEL DATASET SUMMARY ===


,horizon_hours,split,samples,positive_samples,negative_samples,positive_pct
0,1,train,25586,70,25516,0.274
1,1,test,7248,38,7210,0.524
2,2,train,25571,70,25501,0.274
3,2,test,7246,38,7208,0.524
4,3,train,25555,70,25485,0.274
5,3,test,7244,38,7206,0.525
6,4,train,25542,70,25472,0.274
7,4,test,7242,38,7204,0.525
8,5,train,25530,69,25461,0.270
9,5,test,7239,38,7201,0.525



=== FINAL POSITIVE SAMPLES BY SPLIT ===


split,test,train
horizon_hours,,
1,38,70
2,38,70
3,38,70
4,38,70
5,38,69
6,38,69



=== FEATURE SET COMPARISON ===


,metric,original_25_features,final_20_features
0,number_of_features,25.00,20.00
1,complete_rows,21321.00,32881.00
2,incomplete_rows,12550.00,990.00
3,complete_pct,62.95,97.08



Final model dataset saved to:
..\data\processed\recife_2018_2021_model_dataset.csv


## Conclusion

The final supervised-learning dataset was prepared using only meteorological information available at or before each prediction timestamp.

An initial feature audit identified severe data-availability limitations in the wind-related variables. Wind speed, wind gust, wind direction and lagged wind speed were missing from approximately 46% of the training period and caused the loss of a substantial number of severe-rainfall observations.

Because severe rainfall is already a rare class, these wind variables were excluded from the final supervised feature set instead of being artificially imputed.

The final feature set contains 20 leakage-safe predictors based on precipitation, temperature, humidity, atmospheric pressure and temporal information.

A chronological train/test strategy is maintained:

- 2018–2020: training data
- 2021: test data

Rows affected by unavailable meteorological history or unavailable future targets remain excluded from the corresponding modeling task.

The resulting dataset is ready for baseline machine-learning experiments.